In [1]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"


In [3]:
!pip install -q accelerate pillow
!pip install git+https://github.com/huggingface/transformers.git

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-jck1u2_u
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-jck1u2_u
  Resolved https://github.com/huggingface/transformers.git to commit a553395766001116a719c82870171f8d6b458c98
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
# Authenticate with Hugging Face
from huggingface_hub import login

# Method 1: Kaggle Secrets (recommended)
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("Logged in via Kaggle Secrets")
except Exception:
    print("Could not find HF_TOKEN in Kaggle Secrets. Set it up or login manually.")

Logged in via Kaggle Secrets


In [6]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch
import os

local_path = "/kaggle/working/gemma4-e4b"
model_id = "google/gemma-4-e4b-it"

# Check if local cache is valid (has config.json)
if os.path.exists(local_path) and os.path.exists(f"{local_path}/config.json"):
    print("Loading from local cache...")
    source = local_path
else:
    print("Local cache missing. Downloading from HuggingFace...")
    source = model_id

processor = AutoProcessor.from_pretrained(source)
model = AutoModelForImageTextToText.from_pretrained(
    source,
    dtype=torch.bfloat16,
    device_map="auto"
)

if source == model_id:
    print("Saving to local cache...")
    model.save_pretrained(local_path)
    processor.save_pretrained(local_path)
    print("Saved.")

print("Model ready")
print(f"GPU 0 free: {torch.cuda.mem_get_info(0)[0]/1e9:.1f}GB")
print(f"GPU 1 free: {torch.cuda.mem_get_info(1)[0]/1e9:.1f}GB")



Local cache missing. Downloading from HuggingFace...


processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Saving to local cache...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved.
Model ready
GPU 0 free: 13.0GB
GPU 1 free: 2.1GB


In [7]:
from PIL import Image

import time

def test_dashboard(image_path, label):
    image = Image.open(image_path)
    
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": """Describe this data visualization. 
Extract all visible numbers, percentages, and values.
Use this structure:

## Chart Type
[what type of chart is this]

## Key Values
[list every number, percentage, or data value you can see]

## Trends
[any patterns or insights visible]"""}
            ]
        }
    ]
    
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)
    
    start = time.time()
    outputs = model.generate(**inputs, max_new_tokens=2048)
    elapsed = time.time() - start
    
    decoded = processor.decode(outputs[0], skip_special_tokens=True)
    
    print(f"\n{'='*60}")
    print(f"IMAGE: {label} | Time: {elapsed:.1f}s")
    print(f"{'='*60}")
    print(decoded)


# Start with just one
# test_dashboard("/kaggle/input/datasets/shahfazalmohammed/examples-raw/tourisme-powerbi.png", "Tourisme PowerBI")

In [ ]:
import os

image_dir = "/kaggle/input/datasets/shahfazalmohammed/examples-raw"

images = {
    "scatter_all_parties": "scatter-all-parties.png",
    "scatter_gauche_filtered": "scatter-gauche-filtered.png",
    "scatter_gauche_droite": "scatter-gauche-droite-filtered.png",
    "scatter_point_selected": "scatter-point-selected.png",
    "quintile_distrib": "quintile-distrib.png",
    "boxplot_abstention": "boxplot-abstention.png",
    "choropleth_paris": "choropleth-paris.png",
    "choropleth_marseille": "choropleth-marseille.png",
    "choropleth_paris_detailed": "choropleth-paris-detailed.png",
}

for label, filename in images.items():
    path = os.path.join(image_dir, filename)
    test_dashboard(path, label)


In [ ]:
# test boxplot for distribution par bloc
test_dashboard("/kaggle/input/datasets/shahfazalmohammed/examples-raw/boxplot-1.png", "Distrib par bloc")

In [10]:
# run against synthetic quintile 
image_dir = "/kaggle/input/datasets/shahfazalmohammed/stackedbar-synthetic"

images = {
    "stacked_bar_bars_3_001": "stacked_bar_bars_3_001.png",
    "stacked_bar_bars_8_003": "stacked_bar_bars_8_003.png",
    "stacked_bar_dist_even_007": "stacked_bar_dist_even_007.png",
    "stacked_bar_dist_dominant_008": "stacked_bar_dist_dominant_008.png",
    "stacked_bar_dist_tiny_010": "stacked_bar_dist_tiny_010.png",
    "stacked_bar_random_011": "stacked_bar_random_011.png",
}

for label, filename in images.items():
    path = os.path.join(image_dir, filename)
    test_dashboard(path, label)



IMAGE: stacked_bar_bars_3_001 | Time: 103.6s
user
Describe this data visualization. 
Extract all visible numbers, percentages, and values.
Use this structure:

## Chart Type
[what type of chart is this]

## Key Values
[list every number, percentage, or data value you can see]

## Trends
[any patterns or insights visible]
model
## Chart Type
Stacked Bar Chart

## Key Values
**Y-axis Labels (Ranges):**
* Prix 1 864,0-2 064,0 €/m²
* Prix 2 906,0-3 106,0 €/m²
* Prix 2 343,0-2 543,0 €/m²

**X-axis Scale:**
* 0 to 100 (representing percentage %)

**Legend (Colors correspond to categories):**
* Extrême gauche (Red/Dark Pink)
* Gauche (Pink/Light Red)
* Centre (Yellow/Gold)
* Divers (Grey/Silver)
* Droite (Blue)
* Extrême droite (Black)

**Approximate Data Readings (Percentages for each category within each row):**

**Row 1: Prix 1 864,0-2 064,0 €/m²**
* Extrême gauche: ~33%
* Gauche: ~32%
* Centre: ~17%
* Divers: ~15%
* Droite: ~18%
* Extrême droite: ~5%

**Row 2: Prix 2 906,0-3 106,0 €/m²**

In [11]:
# run against synthetic box plots 
image_dir = "/kaggle/input/datasets/shahfazalmohammed/boxplot-synthetic"

images = {
    "boxplot_01_baseline": "boxplot_01_baseline.png",
    "boxplot_02_outliers": "boxplot_02_outliers.png",
    "boxplot_03_tooltips_center": "boxplot_03_tooltips_center.png",
    "boxplot_04_proximity": "boxplot_04_proximity.png",
    "boxplot_05_size_variation": "boxplot_05_size_variation.png",
    "boxplot_06_full_complexity": "boxplot_06_full_complexity.png",
}

for label, filename in images.items():
    path = os.path.join(image_dir, filename)
    test_dashboard(path, label)



IMAGE: boxplot_01_baseline | Time: 90.3s
user
Describe this data visualization. 
Extract all visible numbers, percentages, and values.
Use this structure:

## Chart Type
[what type of chart is this]

## Key Values
[list every number, percentage, or data value you can see]

## Trends
[any patterns or insights visible]
model
## Chart Type
Box Plot

## Key Values
**Y-axis (Note sur 20):**
* 0
* 5
* 10
* 15
* 20

**Data Points/Plot Features (Estimates based on the box plots):**

**Maths:**
* Median (Q2): approx 12.5
* Q1: approx 11.0
* Q3: approx 14.0
* Min (Whisker): approx 7.5
* Max (Whisker): approx 17.5

**Français:**
* Median (Q2): approx 10.5
* Q1: approx 8.0
* Q3: approx 12.5
* Min (Whisker): approx 7.0
* Max (Whisker): approx 15.0

**Histoire:**
* Median (Q2): approx 14.5
* Q1: approx 12.5
* Q3: approx 16.0
* Min (Whisker): approx 6.5
* Max (Whisker): approx 19.5

**Sciences:**
* Median (Q2): approx 10.5
* Q1: approx 7.5
* Q3: approx 15.0
* Min (Whisker): approx 4.0
* Max (Whisker

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/shahfazalmohammed/boxplot-synthetic/stacked_bar_dist_tiny_010.png'

In [12]:
# run against synthetic box plots 
image_dir = "/kaggle/input/datasets/shahfazalmohammed/boxplot-synthetic"

images = {
    "boxplot_05_size_variation": "boxplot_05_size_variation.png",
    "boxplot_06_full_complexity": "boxplot_06_full_complexity.png",
}

for label, filename in images.items():
    path = os.path.join(image_dir, filename)
    test_dashboard(path, label)



IMAGE: boxplot_05_size_variation | Time: 75.9s
user
Describe this data visualization. 
Extract all visible numbers, percentages, and values.
Use this structure:

## Chart Type
[what type of chart is this]

## Key Values
[list every number, percentage, or data value you can see]

## Trends
[any patterns or insights visible]
model
## Chart Type
Box plot

## Key Values
**Y-axis (Note sur 20):** 0, 5, 10, 15, 20
**X-axis Categories:** Maths, Français, Histoire, Sciences, Arts, Sport

**Visual Data Estimates (Approximate Median, Quartiles, Range):**

*   **Maths:**
    *   Minimum: ~2
    *   Q1: ~8.5
    *   Median: ~12.5
    *   Q3: ~15
    *   Maximum: ~20
*   **Français:**
    *   Minimum: ~11.5
    *   Q1: ~11.5
    *   Median: ~12.5
    *   Q3: ~13
    *   Maximum: ~14
*   **Histoire:**
    *   Minimum: ~2
    *   Q1: ~9
    *   Median: ~12.5
    *   Q3: ~14
    *   Maximum: ~18
*   **Sciences:**
    *   Minimum: ~12
    *   Q1: ~13
    *   Median: ~14
    *   Q3: ~14.5
    *   Maxim

In [14]:
# run against synthetic choropleth 
image_dir = "/kaggle/input/datasets/shahfazalmohammed/choropleth-synthetic"

images = {
    "choropleth_01_political": "choropleth_01_political.png",
    "choropleth_02_transport": "choropleth_02_transport.png",
    "choropleth_03_circles": "choropleth_03_circles.png",
    "choropleth_04_political_circles": "choropleth_04_political_circles.png",
    "choropleth_05_gradient": "choropleth_05_gradient.png",
    "choropleth_06_gradient_labels": "choropleth_06_gradient_labels.png",
}

for label, filename in images.items():
    path = os.path.join(image_dir, filename)
    test_dashboard(path, label)


IMAGE: choropleth_01_political | Time: 37.0s
user
Describe this data visualization. 
Extract all visible numbers, percentages, and values.
Use this structure:

## Chart Type
[what type of chart is this]

## Key Values
[list every number, percentage, or data value you can see]

## Trends
[any patterns or insights visible]
model
## Chart Type
A conceptual diagram or map showing the results of an election across different geographical zones, represented by colored hexagons.

## Key Values
There are no visible numbers, percentages, or specific quantitative values on the chart. The only "values" are the labels for the zones (A through L) and the color assignments for the political blocs.

## Trends
The diagram illustrates the distribution of political victory across 12 defined zones (A through L). The color coding suggests a pattern of political dominance by specific blocs in different areas:

*   **Left-leaning blocs (Rouge/Pink/Maroon):** Dominate Zones A, D, E, H, K (Extrême gauche, Gau

In [ ]:
#run this if you ever run into memory/space issues. 
import gc
import torch

del model
gc.collect()
torch.cuda.empty_cache()
print("Memory cleared")
print(f"GPU 0 free: {torch.cuda.mem_get_info(0)[0]/1e9:.1f}GB")
print(f"GPU 1 free: {torch.cuda.mem_get_info(1)[0]/1e9:.1f}GB")
